# 15 · Brand false matches in the text graph

Do a brand's mentions sit next to posts from its own industry? Industries per brand come from `data/processed/brand_industry.csv` (a first pass by an AI agent from general knowledge, not checked by a person). No views are needed, so this also runs on Facebook and Instagram.

In [1]:
import sys, json, warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(ROOT / 'src'))
import eda_utils as eu, brand_mismatch as bm
pd.set_option('display.width', 200)

In [2]:
raw = eu.load(ROOT); info = pd.read_csv(ROOT / 'data' / 'processed' / 'brand_industry.csv'); ind = info.set_index('brand').industry
info.head()

,brand,industry,what_it_is,ambiguous_name,other_meanings,confidence
0,AAA,other,"US auto club: roadside help, travel, insurance",yes,AAA batteries; AAA video-game tier; AAA credit...,medium
1,AB InBev,beverages,"Global brewer (Budweiser, Corona, Michelob)",no,NaN,high
2,AIG,finance and insurance,Global insurance and financial services group,no,NaN,high
3,AMD,tech and software,"Semiconductor maker (CPUs, GPUs)",no,NaN,high
4,AT&T,telecom,US mobile and broadband carrier,no,NaN,high


## Industry mix and flagged names

In [3]:
info.industry.value_counts().to_frame('brands').assign(ambiguous=info.groupby('industry').ambiguous_name.apply(lambda s: (s == 'yes').sum()))

,brands,ambiguous
industry,,
retail and e-commerce,64,6
food and restaurants,46,11
finance and insurance,43,18
tech and software,31,3
telecom,27,11
automotive,23,3
travel and hospitality,23,6
media and entertainment,17,4
consumer goods,13,2


## Neighbour agreement per mention
Share of a mention's other-brand neighbours (10 nearest by TF-IDF cosine) from the same industry, against the share expected by chance.

In [4]:
out = {p: bm.neighbour_agreement(raw, ind, p) for p in raw.platform.unique()}; M = pd.concat([v[0] for v in out.values()])
M.groupby('platform')[['same industry', 'expected']].mean().assign(lift=lambda d: d['same industry'] / d.expected).round(3)

,same industry,expected,lift
platform,,,
Facebook,0.312,0.205,1.521
Instagram,0.290,0.160,1.807
TikTok,0.247,0.087,2.826
X (Twitter),0.243,0.091,2.661
YouTube,0.238,0.089,2.683


## Brands that sit away from their industry
Lift = observed / expected; brands with at least 10 mentions on the platform. Top words: TF-IDF of the brand's mentions minus the rest of the platform.

In [5]:
S = bm.brand_scores(M); R = {eu.NAMES[p]: r for p, (_, r) in out.items()}
low = S.head(20).assign(words=lambda d: [bm.top_terms(R[p], b) for p, b in zip(d.platform, d.brand)]).merge(info[['brand', 'ambiguous_name']], on='brand')
low.round(2)

,platform,brand,industry,mentions,observed,expected,lift,words,ambiguous_name
0,TikTok,AAA,other,14,0.00,0.01,0.00,"aaa, aa, fyp, gu, fortunate, tourist, parati, ...",yes
1,X (Twitter),Frontier,airlines,10,0.00,0.02,0.00,"frontier, ai, models, source, solana, open, an...",yes
2,X (Twitter),Decathlon,sportswear,14,0.00,0.02,0.00,"decathlon, pour, le, leroymerlinfr, losclive, ...",no
3,YouTube,Gap,fashion and apparel,10,0.00,0.03,0.00,"gap, agent, real, market, agents, ____________...",yes
4,X (Twitter),Avanza,other,10,0.00,0.00,0.00,"avanza, på, och, att, har, för, jag, med",yes
5,TikTok,Authentic Brands Group,fashion and apparel,10,0.01,0.03,0.29,"reebok, designed, icecream, questions, ad, sha...",no
6,YouTube,Sling TV,media and entertainment,10,0.02,0.05,0.29,"chardikla, chardiklatimetvlive, dish, tv, char...",yes
7,TikTok,Enterprise,travel and hospitality,10,0.03,0.09,0.32,"enterprise, high, ai, station, global, drone, ...",yes
8,TikTok,Kontoor Brands,fashion and apparel,15,0.01,0.02,0.35,"wrangler, jeep, rubicon, ariat, westernfashion...",no
9,Facebook,Motorola,tech and software,116,0.13,0.33,0.40,"motorola, na, telefon, się, moto, pl, lenovo, nie",no


## Do names with another meaning score lower?
AUC of low lift against the agent's flag, brand-level bootstrap CI. 0.5 means no relation.

In [6]:
A = bm.flag_auc(S, info); A

{'AUC': 0.6150320943424391,
 'low': np.float64(0.5466483057172712),
 'high': np.float64(0.6886376225733921),
 'brand-platform pairs': 424,
 'flagged': 105}

In [7]:
cw = S[S.brand == 'Cricket Wireless']; cw.round(2)

,platform,brand,industry,mentions,observed,expected,lift
644,YouTube,Cricket Wireless,telecom,20,0.06,0.06,1.05
96,TikTok,Cricket Wireless,telecom,20,0.04,0.03,1.40
372,X (Twitter),Cricket Wireless,telecom,10,0.13,0.06,2.04


In [8]:
json.dump({'pairs': len(S), 'flag_auc': A, 'lowest': low.to_dict('records'), 'cricket_wireless': cw.to_dict('records')}, open(ROOT / 'data' / 'processed' / 'mismatch_numbers.json', 'w'), indent=1, default=float)

Takeaway: low agreement finds names used in another sense (e.g. Frontier and Gap in AI posts), but also posts in another language, and it misses Cricket Wireless, whose neighbours are other off-topic posts. Candidates need a hand check before any frequency is claimed.